Step 1 - Setup 

In [1]:
import asyncio
import sys

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from PIL import Image, ImageChops, ImageEnhance
import numpy as np
import io
import os
import gradio as gr
gr.close_all()
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa
import librosa.display

load_dotenv("D:/Deepfake-detection/.env")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # speeds up repeated fixed-size (224x224) inference on GPU
print("Using device:", device)

# ---- Face/image deepfake model ----
model = models.efficientnet_b0(weights=None)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)
model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/efficientnet_best.pth", map_location=device))
model = model.to(device)
model.eval()
class_names = ['fake', 'real']
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# ---- Audio (ASVspoof) deepfake model ----
asv_model = models.efficientnet_b0(weights=None)
asv_num_features = asv_model.classifier[1].in_features
asv_model.classifier[1] = nn.Linear(asv_num_features, 2)
asv_model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/asvspoof_efficientnet_best.pth", map_location=device))
asv_model = asv_model.to(device)
asv_model.eval()
# Index 0 = bonafide (real), index 1 = spoof (fake) - alphabetical ImageFolder order from training
asv_class_names = ['bonafide', 'spoof']
IMG_SIZE_ASV = 224
asv_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_ASV, IMG_SIZE_ASV)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not found in .env - AI explanations will be disabled until you set it.")
client = InferenceClient(api_key=HF_TOKEN, provider="auto") if HF_TOKEN else None
print("All models and clients loaded.")

Using device: cuda
All models and clients loaded.


Step 2 - The three prediction functions

plit into small reusable helper functions (crop_face_from_frame, classify_face_crop) so image/video/webcam all share the same core logic instead of duplicating it three times — cleaner code

Webcam tab intentionally skips the GenAI explanation call — since webcam frames update rapidly, calling the Hugging Face API on every single frame would be slow and could hit rate limits fast; it just shows a quick label instead

In [2]:
import concurrent.futures
import time

# ---- Face detection + image/video/webcam classification ----
def crop_face_from_frame(frame):
    """Detect the largest face at full resolution and return (crop, face_found)."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) > 0:
        faces_sorted = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        x, y, w, h = faces_sorted[0]
        pad = int(0.15 * max(w, h))
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(frame.shape[1], x + w + pad), min(frame.shape[0], y + h + pad)
        return frame[y0:y1, x0:x1], True
    return frame, False


def classify_face_crop(face_crop):
    """Single-image classification (image/webcam tabs)."""
    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(face_rgb)
    input_tensor = transform(pil_img).unsqueeze(0).to(device)
    with torch.inference_mode():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
    return probs.cpu().numpy()


def classify_face_crops_batched(face_crops):
    """Classify multiple face crops in a single GPU forward pass - used by
    the video tab so 5 sampled frames cost one pass instead of five."""
    tensors = []
    for crop in face_crops:
        face_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(face_rgb)
        tensors.append(transform(pil_img))
    batch = torch.stack(tensors).to(device)
    with torch.inference_mode():
        outputs = model(batch)
        probs = torch.softmax(outputs, dim=1)
    return probs.cpu().numpy()  # shape: (num_frames, 2)


def generate_ela_image(pil_image, quality=90):
    """Error Level Analysis: re-save at a known JPEG quality and diff against
    the original. Returns (ela_pil_image, mean_error_score)."""
    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")
    buffer = io.BytesIO()
    pil_image.save(buffer, "JPEG", quality=quality)
    buffer.seek(0)
    resaved = Image.open(buffer)
    ela_image = ImageChops.difference(pil_image, resaved)
    extrema = ela_image.getextrema()
    max_diff = max([ex[1] for ex in extrema]) or 1
    scale_factor = 255.0 / max_diff
    ela_image = ImageEnhance.Brightness(ela_image).enhance(scale_factor)
    mean_error = float(np.array(ela_image).mean())
    return ela_image, round(mean_error, 2)


# ---- Evidence interpreters: turn raw numbers into grounded, plain-language readings ----
def interpret_confidence(confidence):
    if confidence >= 90:
        return "very high confidence - a large, decisive gap between the two class probabilities"
    elif confidence >= 70:
        return "moderate confidence - a reasonably clear gap between classes, but not overwhelming"
    else:
        return "low confidence - the two probabilities are close together, near a 50/50 split"


def interpret_ela(score):
    if score is None:
        return None
    if score < 15:
        return f"a low ELA error level ({score}), consistent with a single clean compression pass - typical of an unedited real photo or frame"
    elif score < 35:
        return f"a moderate ELA error level ({score}) - this can result from normal re-saving or sharing and isn't necessarily a sign of editing on its own"
    else:
        return f"a high, uneven ELA error level ({score}) - a pattern sometimes seen in spliced or re-edited regions, though repeated compression during normal sharing can also cause this"


def _call_explanation_api(prompt, timeout=12):
    """Runs the HF API call with a hard timeout so a slow network call
    can't stall the whole result."""
    def _call():
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3-0324",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        return response.choices[0].message.content

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_call)
        try:
            return future.result(timeout=timeout)
        except concurrent.futures.TimeoutError:
            return "(Explanation timed out - the model's verdict and evidence above are still valid, only the write-up was delayed.)"
        except Exception as e:
            print("EXPLANATION ERROR:", type(e).__name__, str(e))
            return f"(Explanation unavailable right now: {type(e).__name__})"


def generate_explanation(prediction, confidence, fake_prob, real_prob, ela_score=None, face_detected=True, modality="image"):
    """Builds a prompt grounded ONLY in the actual computed evidence for this
    result, and instructs the model to explain the verdict specifically -
    not with generic disclaimers - so the user understands the real 'why'."""
    if client is None:
        return "(Explanation unavailable: no HF_TOKEN configured in your .env file.)"

    confidence_read = interpret_confidence(confidence)
    ela_read = interpret_ela(ela_score)

    evidence_lines = [
        f"- The model classified this {modality} as {prediction.upper()} with {confidence}% confidence ({confidence_read}).",
        f"- Full probability split: {fake_prob}% fake, {real_prob}% real."
    ]
    if ela_read:
        evidence_lines.append(f"- Error Level Analysis reading: {ela_read}.")
    if not face_detected:
        evidence_lines.append("- No face could be automatically detected, so the classifier analyzed the full frame rather than a cropped face - this lowers reliability and must be mentioned.")

    evidence_text = "\n".join(evidence_lines)

    prompt = f"""You are explaining a deepfake-detection result to a non-technical end user who genuinely wants to understand WHY the system reached this verdict - not a generic disclaimer.

Evidence gathered by the pipeline:
{evidence_text}

Write a 3-4 sentence explanation that:
1. States the verdict plainly.
2. Explains, using ONLY the evidence above, what specifically supports that verdict (how decisive the confidence gap is, what the ELA reading suggests if provided, whether face detection succeeded).
3. Notes a real limitation ONLY if it's grounded in the evidence above (e.g. borderline confidence, missing face, ambiguous ELA) - do not invent generic caveats if the evidence is actually strong.
4. Do NOT invent forensic details, artifacts, or reasoning that isn't in the evidence list above.

Be direct, specific, and genuinely informative. Avoid vague filler like "this is just a statistical prediction" unless the evidence actually shows a weak or borderline result."""

    return _call_explanation_api(prompt)


def _format_result(prediction, confidence, fake_prob, real_prob, ela_score, face_found, explanation, extra=""):
    warning = "" if face_found else "\n⚠️ No face detected - analyzed the full frame instead, so treat this result with extra caution.\n"
    ela_line = f"ELA mean error: {ela_score}\n" if ela_score is not None else ""
    return (f"Prediction: {prediction.upper()}\nConfidence: {confidence}%\n{warning}{extra}\n"
            f"Fake: {fake_prob}% | Real: {real_prob}%\n{ela_line}\n"
            f"Explanation:\n{explanation}")


# ---- IMAGE TAB ----
def image_predict(pil_image):
    if pil_image is None:
        return "Please upload an image.", None
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    face_crop, face_found = crop_face_from_frame(img_array)
    probs = classify_face_crop(face_crop)
    pred_idx = int(np.argmax(probs))
    prediction = class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    fake_prob = round(float(probs[0]) * 100, 2)
    real_prob = round(float(probs[1]) * 100, 2)
    ela_image, ela_score = generate_ela_image(pil_image)
    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob, ela_score, face_found, modality="image")
    result = _format_result(prediction, confidence, fake_prob, real_prob, ela_score, face_found, explanation)
    return result, ela_image


# ---- VIDEO TAB (batched classification + single ELA pass) ----
def video_predict(video_path):
    if video_path is None:
        return "Please upload a video.", None
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return "Could not read video.", None

    num_frames = 5
    frame_indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    face_crops = []
    any_face_found = False
    first_frame_rgb = None

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()
        if not success:
            continue
        face_crop, face_found = crop_face_from_frame(frame)
        any_face_found = any_face_found or face_found
        face_crops.append(face_crop)
        if first_frame_rgb is None:
            first_frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    cap.release()

    if len(face_crops) == 0:
        return "No frames could be processed.", None

    all_probs = classify_face_crops_batched(face_crops)  # one GPU pass for all frames
    avg_probs = np.mean(all_probs, axis=0)

    ela_image, ela_score = generate_ela_image(Image.fromarray(first_frame_rgb))

    pred_idx = int(np.argmax(avg_probs))
    prediction = class_names[pred_idx]
    confidence = round(float(avg_probs[pred_idx]) * 100, 2)
    fake_prob = round(float(avg_probs[0]) * 100, 2)
    real_prob = round(float(avg_probs[1]) * 100, 2)
    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob, ela_score, any_face_found, modality="video (averaged across sampled frames)")
    extra = f"(Averaged across {len(face_crops)} sampled frames)\n"
    result = _format_result(prediction, confidence, fake_prob, real_prob, ela_score, any_face_found, explanation, extra)
    return result, ela_image


# ---- WEBCAM TAB ----
def take_snapshot_and_analyze(pil_image):
    """Full analysis (model + ELA + AI explanation) for a single captured snapshot."""
    if pil_image is None:
        return "No frame captured - click the camera icon on the video feed first, then click Analyze.", None

    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    face_crop, face_found = crop_face_from_frame(img_array)
    probs = classify_face_crop(face_crop)
    pred_idx = int(np.argmax(probs))
    prediction = class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    fake_prob = round(float(probs[0]) * 100, 2)
    real_prob = round(float(probs[1]) * 100, 2)
    ela_image, ela_score = generate_ela_image(pil_image)
    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob, ela_score, face_found, modality="webcam snapshot")
    result = _format_result(prediction, confidence, fake_prob, real_prob, ela_score, face_found, explanation)
    return result, ela_image


def webcam_predict(pil_image):
    """Lightweight streaming label with no AI explanation call - runs fast,
    for continuous use. Use Analyze Snapshot for the full ELA + explanation."""
    if pil_image is None:
        return "Waiting for webcam frame..."

    t0 = time.time()
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

    max_dim = 480
    h, w = img_array.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img_array = cv2.resize(img_array, (int(w * scale), int(h * scale)))

    t1 = time.time()
    face_crop, face_found = crop_face_from_frame(img_array)
    t2 = time.time()
    probs = classify_face_crop(face_crop)
    t3 = time.time()

    print(f"Resize: {t1-t0:.2f}s | Face detect: {t2-t1:.2f}s | Classify: {t3-t2:.2f}s | Face found: {face_found}")

    pred_idx = int(np.argmax(probs))
    prediction = class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    tag = "" if face_found else " (no face detected)"
    return f"Prediction: {prediction.upper()} ({confidence}%){tag}"


# ---- AUDIO TAB (real trained ASVspoof EfficientNet model) ----
def audio_to_spectrogram_image(audio_path, sr=16000, n_mels=128):
    """Same preprocessing used to train the ASVspoof EfficientNet model."""
    try:
        y, _ = librosa.load(audio_path, sr=sr)
    except Exception:
        return None
    if len(y) == 0:
        return None

    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.axis('off')
    librosa.display.specshow(mel_spec_db, sr=sr, ax=ax, cmap='magma')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    buf = io.BytesIO()
    fig.savefig(buf, dpi=100)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def classify_audio(audio_path):
    spec_image = audio_to_spectrogram_image(audio_path)
    if spec_image is None:
        return None, None
    input_tensor = asv_transform(spec_image).unsqueeze(0).to(device)
    with torch.inference_mode():
        output = asv_model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
    return probs.cpu().numpy(), spec_image


def audio_predict(audio_path):
    if audio_path is None:
        return "Please upload or record audio.", None

    probs, spec_image = classify_audio(audio_path)
    if probs is None:
        return "Could not process this audio file - it may be empty or in an unsupported format.", None

    pred_idx = int(np.argmax(probs))
    label_map = {0: "real", 1: "fake"}  # bonafide -> real, spoof -> fake
    prediction = label_map[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    real_prob = round(float(probs[0]) * 100, 2)
    fake_prob = round(float(probs[1]) * 100, 2)

    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob, ela_score=None, face_detected=True, modality="audio clip")

    result = (f"Prediction: {prediction.upper()}\nConfidence: {confidence}%\n\n"
              f"Real (bonafide): {real_prob}% | Fake (spoof): {fake_prob}%\n\n"
              f"Explanation:\n{explanation}")
    return result, spec_image

Step 3 - Build the tabbed interface

gr.Blocks(...) — the flexible layout container (vs. the simpler gr.Interface we used before)

gr.Tab(...) — creates separate tabs, each with its own inputs/outputs/logic

gr.Video(...) — Gradio's built-in video upload widget, handles the file automatically

sources=["webcam"] on the image input — tells Gradio to show a "take snapshot from webcam" option instead of just file upload

Each tab has its own Analyze button rather than auto-running on upload — gives the user control over when prediction runs (important for video especially, since it takes a moment to process)

In [3]:
pip install --upgrade gradio

Note: you may need to restart the kernel to use updated packages.


In [3]:
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@600;800&family=Inter:wght@400;600&display=swap');

body, .gradio-container {
    background: linear-gradient(-45deg, #000000, #0a0a0a, #1a1a1a, #050505) !important;
    background-size: 400% 400% !important;
    animation: gradientShift 15s ease infinite !important;
    font-family: 'Inter', sans-serif !important;
    position: relative !important;
}

@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

#particle-bg {
    position: fixed !important;
    top: 0; left: 0;
    width: 100vw !important;
    height: 100vh !important;
    z-index: 0 !important;
    pointer-events: none !important;
}

.gradio-container > * {
    position: relative !important;
    z-index: 1 !important;
}

.gradio-container h1, .gradio-container h1 * {
    font-family: 'Orbitron', sans-serif !important;
    background: linear-gradient(90deg, #ffffff, #999999, #ffffff, #666666, #ffffff);
    background-size: 300% auto;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    animation: shine 6s linear infinite;
    text-shadow: 0 0 40px rgba(255, 255, 255, 0.25);
}

@keyframes shine {
    to { background-position: 300% center; }
}

.gradio-container, .tabs, .tab-nav {
    perspective: 1600px !important;
}

.block {
    background: rgba(255, 255, 255, 0.03) !important;
    backdrop-filter: blur(14px) !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    border-radius: 18px !important;
    transform-style: preserve-3d !important;
    transition: transform 0.4s cubic-bezier(0.23, 1, 0.32, 1), box-shadow 0.4s ease !important;
    animation: floatY 6s ease-in-out infinite;
}

@keyframes floatY {
    0%, 100% { transform: translateY(0px); }
    50% { transform: translateY(-6px); }
}

.block:hover {
    box-shadow:
        0 25px 50px rgba(0, 0, 0, 0.7),
        0 0 40px rgba(255, 255, 255, 0.15),
        inset 0 0 20px rgba(255, 255, 255, 0.04) !important;
    border: 1px solid rgba(255, 255, 255, 0.4) !important;
    animation-play-state: paused;
}

button.primary, button.secondary, button.stop {
    position: relative !important;
    border: none !important;
    border-radius: 14px !important;
    font-weight: 700 !important;
    letter-spacing: 0.5px;
    color: #ffffff !important;
    transform-style: preserve-3d !important;
    transition: transform 0.25s cubic-bezier(0.23, 1, 0.32, 1), box-shadow 0.25s ease !important;
    box-shadow:
        0 6px 0 rgba(0, 0, 0, 0.6),
        0 10px 20px rgba(0, 0, 0, 0.5) !important;
}

button.primary {
    background: linear-gradient(135deg, #1a1a1a, #000000) !important;
    border: 1px solid rgba(255, 255, 255, 0.15) !important;
    animation: pulseGlow 2.5s ease-in-out infinite;
}

button.stop {
    background: linear-gradient(135deg, #2b0000, #000000) !important;
    border: 1px solid rgba(255, 80, 80, 0.3) !important;
}

button.secondary {
    background: linear-gradient(135deg, #2a2a2a, #0d0d0d) !important;
    border: 1px solid rgba(255, 255, 255, 0.15) !important;
}

@keyframes pulseGlow {
    0%, 100% { box-shadow: 0 6px 0 rgba(0,0,0,0.6), 0 10px 20px rgba(0,0,0,0.5), 0 0 0px rgba(255,255,255,0.3); }
    50% { box-shadow: 0 6px 0 rgba(0,0,0,0.6), 0 10px 20px rgba(0,0,0,0.5), 0 0 30px rgba(255,255,255,0.5); }
}

button.primary:hover, button.secondary:hover, button.stop:hover {
    transform: translateY(-8px) rotateX(12deg) scale(1.06) !important;
    box-shadow:
        0 16px 0 rgba(0, 0, 0, 0.55),
        0 26px 40px rgba(0, 0, 0, 0.6),
        0 0 45px rgba(255, 255, 255, 0.4) !important;
}

button.primary:active, button.secondary:active, button.stop:active {
    transform: translateY(-1px) rotateX(2deg) scale(0.97) !important;
    box-shadow: 0 2px 0 rgba(0, 0, 0, 0.5), 0 4px 10px rgba(0, 0, 0, 0.4) !important;
}

.tab-nav button {
    color: rgba(255, 255, 255, 0.6) !important;
    font-weight: 600 !important;
    transition: all 0.3s cubic-bezier(0.23, 1, 0.32, 1) !important;
    position: relative !important;
}

.tab-nav button:hover {
    transform: translateY(-4px) scale(1.08);
    color: #ffffff !important;
    text-shadow: 0 0 16px rgba(255, 255, 255, 0.8);
}

.tab-nav button.selected {
    color: #ffffff !important;
    text-shadow: 0 0 20px rgba(255, 255, 255, 0.7);
    border-bottom: 2px solid #ffffff !important;
}

textarea, .image-container, .image-frame {
    background: rgba(0, 0, 0, 0.4) !important;
    color: #f0f0f0 !important;
    border-radius: 16px !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    transition: transform 0.4s cubic-bezier(0.23, 1, 0.32, 1), box-shadow 0.4s ease !important;
}

textarea:hover, .image-container:hover, .image-frame:hover {
    transform: translateY(-6px) scale(1.01) !important;
    box-shadow: 0 20px 40px rgba(0, 0, 0, 0.6), 0 0 30px rgba(255, 255, 255, 0.2) !important;
}

textarea:focus {
    box-shadow: 0 0 0 2px #ffffff, 0 0 25px rgba(255, 255, 255, 0.4) !important;
}
"""

combined_js = """
function() {
    // ---- Animated particle-network background ----
    const canvas = document.createElement('canvas');
    canvas.id = 'particle-bg';
    document.body.prepend(canvas);
    const ctx = canvas.getContext('2d');

    function resize() {
        canvas.width = window.innerWidth;
        canvas.height = window.innerHeight;
    }
    resize();
    window.addEventListener('resize', resize);

    const NUM_PARTICLES = 70;
    const MAX_DIST = 140;
    const particles = [];

    for (let i = 0; i < NUM_PARTICLES; i++) {
        particles.push({
            x: Math.random() * canvas.width,
            y: Math.random() * canvas.height,
            vx: (Math.random() - 0.5) * 0.4,
            vy: (Math.random() - 0.5) * 0.4,
            r: Math.random() * 1.5 + 0.5
        });
    }

    function animate() {
        ctx.clearRect(0, 0, canvas.width, canvas.height);

        for (let p of particles) {
            p.x += p.vx;
            p.y += p.vy;
            if (p.x < 0 || p.x > canvas.width) p.vx *= -1;
            if (p.y < 0 || p.y > canvas.height) p.vy *= -1;

            ctx.beginPath();
            ctx.arc(p.x, p.y, p.r, 0, Math.PI * 2);
            ctx.fillStyle = 'rgba(255, 255, 255, 0.6)';
            ctx.fill();
        }

        for (let i = 0; i < particles.length; i++) {
            for (let j = i + 1; j < particles.length; j++) {
                const dx = particles[i].x - particles[j].x;
                const dy = particles[i].y - particles[j].y;
                const dist = Math.sqrt(dx * dx + dy * dy);
                if (dist < MAX_DIST) {
                    ctx.beginPath();
                    ctx.moveTo(particles[i].x, particles[i].y);
                    ctx.lineTo(particles[j].x, particles[j].y);
                    ctx.strokeStyle = `rgba(255, 255, 255, ${0.15 * (1 - dist / MAX_DIST)})`;
                    ctx.lineWidth = 0.6;
                    ctx.stroke();
                }
            }
        }

        requestAnimationFrame(animate);
    }
    animate();

    // ---- Mouse-tracked 3D tilt on cards ----
    function addTilt(el) {
        el.addEventListener('mousemove', (e) => {
            const rect = el.getBoundingClientRect();
            const x = e.clientX - rect.left;
            const y = e.clientY - rect.top;
            const centerX = rect.width / 2;
            const centerY = rect.height / 2;
            const rotateX = ((y - centerY) / centerY) * -8;
            const rotateY = ((x - centerX) / centerX) * 8;
            el.style.transform = `perspective(1000px) rotateX(${rotateX}deg) rotateY(${rotateY}deg) translateZ(10px)`;
        });
        el.addEventListener('mouseleave', () => {
            el.style.transform = 'perspective(1000px) rotateX(0deg) rotateY(0deg) translateZ(0px)';
        });
    }

    function applyToAll() {
        document.querySelectorAll('.block').forEach(addTilt);
    }

    applyToAll();
    const observer = new MutationObserver(applyToAll);
    observer.observe(document.body, { childList: true, subtree: true });
}
"""

with gr.Blocks(title="AI-Powered Deepfake Detection System", css=custom_css, js=combined_js) as demo:
    gr.Markdown("# 🎭 AI-Powered Deepfake Detection System")
    gr.Markdown("Detect deepfakes in images, videos, live webcam snapshots, and audio - with Error Level Analysis, mel-spectrogram visualization, and an AI-written explanation grounded in real evidence for every result.")

    with gr.Tab("📷 Image"):
        img_input = gr.Image(type="pil", label="Upload an image")
        img_button = gr.Button("Analyze Image", variant="primary")
        with gr.Row():
            img_output = gr.Textbox(label="Result", lines=12)
            img_ela_output = gr.Image(label="Error Level Analysis")
        img_button.click(fn=image_predict, inputs=img_input, outputs=[img_output, img_ela_output])

    with gr.Tab("🎥 Video"):
        vid_input = gr.Video(label="Upload a video")
        vid_button = gr.Button("Analyze Video", variant="primary")
        with gr.Row():
            vid_output = gr.Textbox(label="Result", lines=12)
            vid_ela_output = gr.Image(label="Error Level Analysis (first sampled frame)")
        vid_button.click(fn=video_predict, inputs=vid_input, outputs=[vid_output, vid_ela_output])

    with gr.Tab("📹 Webcam"):
        gr.Markdown("Click the camera icon on the video feed below to capture a photo, then click Analyze. If lighting is poor or your face is at an angle, detection may fail - the result will flag this.")
        webcam_input = gr.Image(type="pil", label="Webcam", sources=["webcam"])
        with gr.Row():
            snapshot_button = gr.Button("📸 Analyze Snapshot", variant="primary")
            stop_button = gr.Button("🛑 Stop Camera", variant="stop")
        with gr.Row():
            webcam_output = gr.Textbox(label="Result", lines=12)
            webcam_ela_output = gr.Image(label="Error Level Analysis")
        snapshot_button.click(fn=take_snapshot_and_analyze, inputs=webcam_input, outputs=[webcam_output, webcam_ela_output])
        stop_button.click(fn=lambda: (None, None), inputs=None, outputs=[webcam_input, webcam_ela_output])

    with gr.Tab("🎙️ Audio"):
        gr.Markdown("Upload or record audio - classified by a trained EfficientNet-B0 model (ASVspoof-based) as real or AI-generated/cloned speech, using the same mel-spectrogram pipeline it was trained on.")
        audio_input = gr.Audio(type="filepath", label="Upload or record audio")
        audio_button = gr.Button("Analyze Audio", variant="primary")
        with gr.Row():
            audio_output = gr.Textbox(label="Result", lines=12)
            audio_spec_output = gr.Image(label="Mel-Spectrogram (model input)")
        audio_button.click(fn=audio_predict, inputs=audio_input, outputs=[audio_output, audio_spec_output])

demo.launch()

C:\Users\ACER\AppData\Local\Temp\ipykernel_21240\1153300263.py:250: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css, js. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AI-Powered Deepfake Detection System", css=custom_css, js=combined_js) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
client = InferenceClient(api_key=HF_TOKEN, provider="auto")
print("Client recreated.")

In [5]:
try:
    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3-0324",
        messages=[{"role": "user", "content": "Say hello in one sentence."}],
        max_tokens=50
    )
    print(response.choices[0].message.content)
except Exception as e:
    print("ERROR:", type(e).__name__, str(e))

"Hello!" 😊


In [6]:
test = generate_explanation("fake", 65.36, 65.36, 34.64)
print(test)

This result suggests the content is likely a deepfake, with a 65.36% probability of being fake and 34.64% of being real. However, this is a statistical prediction, not a definitive judgment—there’s still a chance it could be authentic. For critical decisions, consider additional verification or context.


In [7]:
model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/efficientnet_finetuned.pth", map_location=device))
model.eval()
print("Loaded fine-tuned model.")

Loaded fine-tuned model.
